In [15]:
import pandas as pd
import dataclasses
from dataclasses import dataclass
import json
from kafka import KafkaProducer
from models import Ride,ride_from_row, green_ride_from_row,ride_serializer, green_ride_from_row

In [16]:
url='https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'

In [17]:
# columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
columns = ['lpep_pickup_datetime',
'lpep_dropoff_datetime',
'PULocationID',
'DOLocationID',
'passenger_count',
'trip_distance',
'tip_amount',
'total_amount']
df = pd.read_parquet(url, columns=columns)

In [18]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [19]:
df.dtypes

lpep_pickup_datetime     datetime64[us]
lpep_dropoff_datetime    datetime64[us]
PULocationID                      int32
DOLocationID                      int32
passenger_count                 float64
trip_distance                   float64
tip_amount                      float64
total_amount                    float64
dtype: object

In [20]:
df = df.astype({'lpep_pickup_datetime': 'str', 'lpep_dropoff_datetime': 'str'})

In [21]:
df

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12
...,...,...,...,...,...,...,...,...
49411,2025-10-31 23:02:00,2025-11-01 00:28:33,241,61,NaN,20.09,0.00,63.84
49412,2025-10-31 23:51:34,2025-11-01 00:20:58,53,225,NaN,10.11,0.00,34.76
49413,2025-10-31 23:08:00,2025-10-31 23:42:00,7,170,NaN,4.20,10.03,60.17
49414,2025-10-31 23:45:00,2025-11-01 00:08:00,255,25,NaN,4.20,4.86,37.29


In [22]:


server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49416 entries, 0 to 49415
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   lpep_pickup_datetime   49416 non-null  str    
 1   lpep_dropoff_datetime  49416 non-null  str    
 2   PULocationID           49416 non-null  int32  
 3   DOLocationID           49416 non-null  int32  
 4   passenger_count        44401 non-null  float64
 5   trip_distance          49416 non-null  float64
 6   tip_amount             49416 non-null  float64
 7   total_amount           49416 non-null  float64
dtypes: float64(4), int32(2), str(2)
memory usage: 4.4 MB


In [24]:
# We encoutered issue with NaN value on the consumer side when running flink, 
# so as a TEMP SOLUTION we will fill NaN values with 0, but in production we should handle it properly
df = df.fillna(0)

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49416 entries, 0 to 49415
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   lpep_pickup_datetime   49416 non-null  str    
 1   lpep_dropoff_datetime  49416 non-null  str    
 2   PULocationID           49416 non-null  int32  
 3   DOLocationID           49416 non-null  int32  
 4   passenger_count        49416 non-null  float64
 5   trip_distance          49416 non-null  float64
 6   tip_amount             49416 non-null  float64
 7   total_amount           49416 non-null  float64
dtypes: float64(4), int32(2), str(2)
memory usage: 4.4 MB


In [26]:
# Now send 1000 records
import time
topic_name = 'green-trips'
t0 = time.time()

for _, row in df.iterrows():
    ride = green_ride_from_row(row)
    producer.send(topic_name, value=ride)
    # print(f"Sent: {ride}")
    # time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 9.91 seconds


In [27]:
#AA: Resume frome here